# Семинар: текстовое ранжирование, часть I

Сегодня научимся ранжировать документы по текстам запроса и документа — без машинного обучения, только формулами.
И главное — научимся честно измерять, стало ли лучше.

План:
1. Данные: вопросы к Википедии и оценки релевантности.
2. Сначала измеряем: Recall@10, MRR@10, P@10.
3. TF-IDF руками: обратный индекс с частотами, IDF, скоринг только по постинг-листам (TaaT), топ-K через кучу.
4. То же в `sklearn`: модель векторного пространства и варианты TF-IDF.
5. Русская морфология*: стемминг, лемматизация, символьные n-граммы, стоп-слова.
6. Кворум: мягкое И для отбора кандидатов.
7. `whoosh`: TF-IDF против BM25F.
8. Лексический разрыв: псевдорелевантная обратная связь (Rocchio) и LSA.
9. Задания*.

Перед запуском положите рядом с ноутбуком файлы `docs.tsv`, `queries.tsv` и `qrels.tsv` (их собирает `prepare_data.py`).

In [2]:
# ! pip3 install -r requirements.txt

## 0. Данные

Коллекция — абзацы русской Википедии из датасета RuBQ (вопросы по Википедии, в IR-формате он входит в бенчмарк ruMTEB).
Мы взяли 15 тыс. абзацев: все, которые размечены как релевантные, плюс случайные остальные.

Запросы — живые вопросы вида «Кто написал…». К каждому запросу есть **qrels** — список абзацев, в которых есть ответ.

В отличие от прошлого семинара тексты **не предобработаны**: регистр, пунктуация и словоформы на месте. Разбираться с ними будем сами.

In [42]:
import os

DOCS_TSV, QUERIES_TSV, QRELS_TSV = "docs.tsv", "queries.tsv", "qrels.tsv"

for path in (DOCS_TSV, QUERIES_TSV, QRELS_TSV):
    assert os.path.exists(path), f"Положите {path} рядом с ноутбуком"

docs = {}  # docid -> текст абзаца
with open(DOCS_TSV, encoding="utf-8") as f:
    for line in f:
        docid, text = line.rstrip("\n").split("\t")
        docs[docid] = text

print(f"{len(docs)} документов, {os.path.getsize(DOCS_TSV) / 2**20:.0f} МБ\n")
for docid in list(docs)[:2]:
    print(docid, docs[docid][:200], "...\n")

15010 документов, 12 МБ

6 В ноябре 2003 года вместо Валерия Газзаева у руля команды встал португальский специалист Артур Жорже, под руководством нового тренера команда одержала победу 3:1 в матче на Суперкубок России против мо ...

7 После окончания сезона 2017/18 завершили карьеры Алексей и Василий Березуцкие, Сергей Игнашевич вошел в тренерский штаб команды, закончились контракты у Понтуса Вернблума, Бибраса Натхо и Георги Милан ...



Формат `queries.tsv`: `qid`, часть выборки (`dev` или `test`), текст запроса.
На `dev` будем подбирать параметры, на `test` — только проверять. Если подбирать и мерить на одних и тех же запросах, мы переобучимся под них и обманем сами себя.

Формат `qrels.tsv`: `qid`, `docid` релевантного документа.

In [43]:
from collections import Counter, defaultdict

queries, split = {}, {}  # qid -> текст, qid -> dev/test
with open(QUERIES_TSV, encoding="utf-8") as f:
    for line in f:
        qid, part, text = line.rstrip("\n").split("\t")
        queries[qid], split[qid] = text, part

qrels = defaultdict(set)  # qid -> множество релевантных docid
with open(QRELS_TSV, encoding="utf-8") as f:
    for line in f:
        qid, docid = line.split()
        qrels[qid].add(docid)

dev = [qid for qid in queries if split[qid] == "dev"]
test = [qid for qid in queries if split[qid] == "test"]
print(f"запросов: {len(queries)} (dev {len(dev)}, test {len(test)})\n")

for qid in ("1", "414"):
    print(f"запрос {qid}: {queries[qid]}")
    for docid in sorted(qrels[qid])[:2]:
        print(f"   релевантен {docid}: {docs[docid][:150]} ...")
    print()

запросов: 1692 (dev 400, test 1292)

запрос 1: Кто написал роман «Хижина дяди Тома»?
   релевантен 35652: «Хижина дяди Тома» (англ. Uncle Tom's Cabin) — роман Гарриет Бичер-Стоу 1852 года, направленный против рабовладения в Америке. Роман произвёл большой  ...

запрос 414: Кто написал картину «Последний день Помпеи»?
   релевантен 18191: Когда работа над картиной была окончательно завершена, Карл Брюллов стал показывать её в Риме в своей мастерской на Via di San Claudio. Заказчик карти ...
   релевантен 18193: «После́дний день Помпе́и» — крупноформатная картина русского художника Карла Брюллова (1799—1852), работа над которой была завершена в 1833 году. Хран ...



In [44]:
import numpy as np

lengths = [len(text.split()) for text in docs.values()]
print(f"длина абзаца в словах: медиана {np.median(lengths):.0f}, среднее {np.mean(lengths):.0f}, максимум {max(lengths)}")
print("релевантных документов на запрос:", sorted(Counter(len(rel) for rel in qrels.values()).items()))

длина абзаца в словах: медиана 48, среднее 62, максимум 1536
релевантных документов на запрос: [(1, 923), (2, 495), (3, 176), (4, 91), (5, 4), (6, 1), (7, 2)]


Абзацы короткие, а у большинства запросов всего 1–2 релевантных документа. Задача похожа на то, что делает поиск в RAG-системе:
найти кусок текста, где лежит ответ на вопрос.

## 1. Сначала измеряем

Прежде чем что-то улучшать, договоримся, как мерить. Пользователь видит только топ, поэтому все метрики считаем по топ-10:

- **Recall@10** — какая доля релевантных документов попала в топ-10;
- **MRR@10** — `1 / позиция` первого релевантного документа (0, если его нет в топ-10), усреднённая по запросам. Чувствительна к тому, *где* в топе стоит ответ;
- **P@10** — доля релевантных среди первых 10. У нас на запрос 1–2 релевантных документа, так что P@10 почти никогда не бывает больше 0.2.

Подробно про метрики (MAP, nDCG, градуированная релевантность) будет в части II.

Ранкер для нас — любая функция «текст запроса → список docid по убыванию скора».

In [45]:
import random


def evaluate(ranker, qids, k=10):
    """Средние Recall@k, MRR@k и P@k ранкера по списку запросов."""
    recall = mrr = precision = 0.0
    for qid in qids:
        top = ranker(queries[qid])[:k]
        relevant = qrels[qid]
        hits = [docid in relevant for docid in top]
        recall += sum(hits) / len(relevant)
        precision += sum(hits) / k
        mrr += next((1 / (pos + 1) for pos, hit in enumerate(hits) if hit), 0.0)
    n = len(qids)
    return {"Recall@10": round(recall / n, 3), "MRR@10": round(mrr / n, 3), "P@10": round(precision / n, 3)}


doc_ids = list(docs)
rng = random.Random(0) # np.random.sample()


def random_ranker(query):
    return rng.sample(doc_ids, 10)


results = {}  # сюда складываем метрики всех методов на test
results["случайная выдача"] = evaluate(random_ranker, test)
results["случайная выдача"]

{'Recall@10': 0.002, 'MRR@10': 0.002, 'P@10': 0.0}

Случайная выдача не находит почти ничего: 10 документов из 15 тыс. Это нижняя граница, от которой будем отталкиваться.

## 2. TF-IDF руками

### 2.1 Обратный индекс с частотами

В прошлый раз постинг-лист хранил только docid. Для ранжирования нужно больше: сколько раз термин встретился в документе (**tf**).
Поэтому постинг теперь — пара `(номер документа, tf)`.

Токенизацию сделаем такой же, как в `sklearn` по умолчанию: слова от двух символов в нижнем регистре. Так потом сможем сверить результаты.

In [46]:
import re
import time

TOKEN_RE = re.compile(r"(?u)\b\w\w+\b")


def tokenize(text):
    return TOKEN_RE.findall(text.lower())


def build_tf_index(texts):
    index = defaultdict(list)  # термин -> [(номер документа, tf), ...]
    for doc_num, text in enumerate(texts):
        for term, tf in Counter(tokenize(text)).items():
            index[term].append((doc_num, tf))
    return dict(index)


texts = [docs[docid] for docid in doc_ids]  # документы нумеруем по порядку, doc_num -> doc_ids[doc_num]

start = time.time()
tf_index = build_tf_index(texts)
print(f"индекс построен за {time.time() - start:.1f} сек, терминов: {len(tf_index)}")
print("постинг-лист 'цунами':", tf_index["цунами"])

индекс построен за 0.5 сек, терминов: 139646
постинг-лист 'цунами': [(1788, 1), (9224, 2), (9225, 3), (9227, 2)]


In [47]:
tf_index['март']

[(407, 1),
 (1269, 1),
 (1471, 1),
 (3088, 1),
 (3291, 1),
 (3782, 1),
 (4107, 1),
 (4575, 1),
 (5338, 1),
 (5585, 1),
 (5702, 1),
 (7209, 1),
 (7255, 1),
 (8021, 1),
 (8741, 1),
 (9640, 1),
 (9747, 1),
 (11599, 1),
 (12400, 1),
 (13022, 1),
 (13745, 1),
 (13753, 2),
 (14677, 1)]

### 2.2 IDF

Документная частота `df` — это просто длина постинг-листа. Её мы получаем бесплатно.

Две формулы IDF:
- классическая: `idf = log(N / df)`;
- сглаженная, как в `sklearn`: `idf = 1 + ln((N + 1) / (df + 1))`. Единица в конце нужна, чтобы термин, который есть во всех документах, не обнулялся полностью.

In [48]:
import math

N = len(texts)
df = {term: len(postings) for term, postings in tf_index.items()}
idf = {term: 1 + math.log((N + 1) / (df_t + 1)) for term, df_t in df.items()}  # как в sklearn

for term in ("году", "был", "который", "пушкин", "цунами"):
    print(f"{term:>10}: df = {df[term]:>5}   log(N/df) = {math.log(N / df[term]):.2f}   sklearn idf = {idf[term]:.2f}")

      году: df =  3133   log(N/df) = 1.57   sklearn idf = 2.57
       был: df =  2594   log(N/df) = 1.76   sklearn idf = 2.76
   который: df =   948   log(N/df) = 2.76   sklearn idf = 3.76
    пушкин: df =    37   log(N/df) = 6.01   sklearn idf = 6.98
    цунами: df =     4   log(N/df) = 8.23   sklearn idf = 9.01


Частые слова вроде «году» и «был» получают маленький вес, редкие «пушкин» и «цунами» — большой. Это и есть закон Ципфа в действии:
немногие частые слова встречаются везде и почти ничего не говорят о документе.

In [49]:
# tf_index

### 2.3 Скоринг: считаем только то, что нужно

Косинус между запросом и документом: `score(q, d) = Σ_t w(t, q) · w(t, d) / |d|`, где `w = tf · idf`.

Ключевое наблюдение: слагаемые ненулевые **только для терминов запроса**. Значит, документы без слов запроса можно даже не трогать.
Идём по постинг-листам терминов запроса и копим скоры в словаре-аккумуляторе. Это схема **term-at-a-time**.

Норму документа `|d|` от запроса не зависит, её считаем один раз заранее, в оффлайне.
Норму запроса можно не считать вовсе: на порядок документов она не влияет.

In [50]:
import heapq

# оффлайн: L2-нормы документов
doc_norm = [0.0] * N
for term, postings in tf_index.items():
    for doc_num, tf in postings:
        doc_norm[doc_num] += (tf * idf[term]) ** 2
doc_norm = [math.sqrt(x) or 1.0 for x in doc_norm]  # один абзац в коллекции состоит из цифры "1" -- без токенов


def taat_scores(query):
    """Term-at-a-time: обходим только постинг-листы терминов запроса."""
    acc = defaultdict(float)
    for term, qtf in Counter(tokenize(query)).items():
        if term not in tf_index:
            continue
        w_q = qtf * idf[term]
        for doc_num, tf in tf_index[term]:
            acc[doc_num] += w_q * tf * idf[term]
    return acc


def manual_tfidf_ranker(query, k=100):
    acc = taat_scores(query)
    # топ-K через кучу: O(n log K) вместо полной сортировки
    top = heapq.nlargest(k, acc.items(), key=lambda item: item[1] / doc_norm[item[0]])
    return [doc_ids[doc_num] for doc_num, _ in top]


for qid in ("1", "414"):
    print(queries[qid])
    for pos, docid in enumerate(manual_tfidf_ranker(queries[qid])[:3], 1):
        mark = "+" if docid in qrels[qid] else " "
        print(f"  {pos}. [{mark}] {docs[docid][:110]}")
    print()

Кто написал роман «Хижина дяди Тома»?
  1. [+] «Хижина дяди Тома» (англ. Uncle Tom's Cabin) — роман Гарриет Бичер-Стоу 1852 года, направленный против рабовла
  2. [ ] Роман «Хижина дяди Тома» был опубликован в 1851—1852 годах в аболиционистском журнале The National Era, отдель
  3. [ ] Относительно тезиса о принадлежности к традиции так называемого мифа о Старом Юге, вполне естественной для пис

Кто написал картину «Последний день Помпеи»?
  1. [+] Художник Михаил Железнов в своих более поздних воспоминаниях писал, что Карл Брюллов рассказывал ему о том, чт
  2. [+] Когда работа над картиной была окончательно завершена, Карл Брюллов стал показывать её в Риме в своей мастерск
  3. [ ] Свой последний фильм «Семь криков в океане» он выпустил в 1986 году. Критики довольно прохладно встретили карт



`[+]` отмечает релевантные документы. Теперь сравним со «скорингом в лоб», где считаем скор для **каждого** документа коллекции.

In [51]:
doc_tfs = [Counter(tokenize(text)) for text in texts]  # прямой индекс: документ -> tf терминов


def bruteforce_scores(query):
    q_terms = Counter(tokenize(query))
    scores = []
    for doc_num in range(N):
        s = sum(qtf * idf.get(t, 0) * doc_tfs[doc_num].get(t, 0) * idf.get(t, 0) for t, qtf in q_terms.items())
        scores.append(s / doc_norm[doc_num])
    return scores


sample = test[:100]

start = time.time()
for qid in sample:
    bruteforce_scores(queries[qid])
brute_time = time.time() - start

start = time.time()
touched = [len(taat_scores(queries[qid])) for qid in sample]
taat_time = time.time() - start

print(f"в лоб:          {brute_time / len(sample) * 1000:.1f} мс на запрос, {N} документов")
print(f"term-at-a-time: {taat_time / len(sample) * 1000:.1f} мс на запрос, в среднем {np.mean(touched):.0f} документов")

в лоб:          16.3 мс на запрос, 15010 документов
term-at-a-time: 0.3 мс на запрос, в среднем 2004 документов


Даже на игрушечной коллекции разница в десятки раз. Но term-at-a-time всё равно трогает в среднем 2 тыс. документов из 15 тыс.:
в запросах много частых слов («кто», «какой», «году»), и их постинг-листы тянут за собой тысячи документов.
Отсюда две идеи: отбрасывать кандидатов **кворумом** (раздел 5) и не досчитывать документы,
которые заведомо не попадут в топ (MaxScore и WAND с лекции).

Теперь качество.

In [52]:
results["TF-IDF руками"] = evaluate(manual_tfidf_ranker, test)
results["TF-IDF руками"]

{'Recall@10': 0.487, 'MRR@10': 0.385, 'P@10': 0.082}

## 3. Модель векторного пространства в sklearn

`TfidfVectorizer` делает всё то же самое: строит матрицу «документ × термин» с весами TF-IDF и нормирует строки по L2.
Тогда скалярное произведение векторов запроса и документа — это косинус.

In [53]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()  # по умолчанию: raw tf, сглаженный idf, L2-нормализация
X = vectorizer.fit_transform(texts)
print(f"матрица {X.shape[0]} x {X.shape[1]}, ненулевых элементов {X.nnz:,} ({X.nnz / (X.shape[0] * X.shape[1]):.5%})")


def top_k(scores, k=100):
    """Индексы k самых больших скоров по убыванию, без полной сортировки."""
    idx = np.argpartition(-scores, k)[:k]
    return idx[np.argsort(-scores[idx])]


def make_ranker(vectorizer, X, prep=lambda text: text):
    def ranker(query, k=100):
        scores = (X @ vectorizer.transform([prep(query)]).T).toarray().ravel()
        return [doc_ids[i] for i in top_k(scores, k)]
    return ranker


sklearn_ranker = make_ranker(vectorizer, X)

# сверяем с ручной реализацией: топ-10 должен совпасть
same = sum(sklearn_ranker(queries[qid])[:10] == manual_tfidf_ranker(queries[qid])[:10] for qid in test[:200])
print(f"топ-10 совпал с ручной версией на {same} запросах из 200")

матрица 15010 x 139646, ненулевых элементов 730,345 (0.03484%)
топ-10 совпал с ручной версией на 197 запросах из 200


In [54]:
X.toarray().shape # (15010, 139646) 

(15010, 139646)

Если где-то топ не совпал, это почти всегда документы с одинаковым скором, которые две реализации упорядочили по-разному.

У векторизатора много параметров, и каждый — это ровно один из вариантов TF-IDF с лекции:

- `sublinear_tf=True` — вместо `tf` берём `1 + log(tf)`: второе вхождение слова даёт меньше, чем первое;
- `binary=True` — `tf` заменяется на 0/1;
- `use_idf=False` — без IDF, только частоты;
- `norm=None` — без нормализации длины: чистое скалярное произведение вместо косинуса;
- `smooth_idf=False` — IDF без сглаживания.

Посмотрим, как они влияют на качество.

In [55]:
import pandas as pd

variants = {
    "по умолчанию": {},
    "sublinear_tf": {"sublinear_tf": True},
    "binary tf": {"binary": True},
    "без idf": {"use_idf": False},
    "без нормализации": {"norm": None},
    "без сглаживания idf": {"smooth_idf": False},
}

rows = {}
for name, params in variants.items():
    vec = TfidfVectorizer(**params)
    rows[name] = evaluate(make_ranker(vec, vec.fit_transform(texts)), test)

pd.DataFrame(rows).T

,Recall@10,MRR@10,P@10
по умолчанию,0.487,0.384,0.082
sublinear_tf,0.495,0.379,0.083
binary tf,0.452,0.327,0.074
без idf,0.277,0.213,0.045
без нормализации,0.460,0.344,0.078
без сглаживания idf,0.492,0.390,0.082


Что видно:
- **IDF — главный ингредиент.** Без него Recall@10 падает с 0.487 до 0.277: частые слова вопроса («кто», «какой») забивают выдачу;
- **нормализация длины тоже важна**: без неё длинные абзацы выигрывают просто потому, что в них больше слов (0.46 против 0.487);
- бинарный tf хуже: повторы слова что-то значат. Сублинейный `1 + log(tf)` и обычный tf здесь почти равны — абзацы короткие, больших tf в них мало.

Дальше по умолчанию используем `sublinear_tf=True`.

## 4. Русская морфология

Запрос «Кто написал роман…», а в документе «роман написан», «автором романа». Для TF-IDF это разные термины.
Попробуем три способа склеить словоформы:

- **стемминг** (Snowball): отрезаем окончание по правилам;
- **лемматизация** (`pymorphy3`): приводим слово к словарной форме;
- **символьные n-граммы**: режем слова на куски по 4 символа. Про морфологию не знаем ничего, но у «роман», «романа», «романом» много общих кусков.

Но сначала одна ловушка. В Википедии у названий часто стоят **знаки ударения**: «Камбо́джа», «После́дний день Помпе́и».
Ударение — отдельный символ Unicode (combining acute, `U+0301`), и регулярка `\w+` его словом не считает: «Камбо́джа» превращается в два токена «камбо» и «джа».
Такой абзац не найдётся по запросу «Камбоджа», и это не лексический разрыв, а баг токенизации. Поэтому перед нормализацией словоформ убираем ударения и заодно заменяем «ё» на «е».

In [57]:
from functools import lru_cache

import unicodedata

import pymorphy3
from nltk.stem.snowball import SnowballStemmer

morph = pymorphy3.MorphAnalyzer()
stemmer = SnowballStemmer("russian")
WORD_RE = re.compile(r"\w+")
ACCENTS = {"\u0301", "\u0300"}  # знаки ударения


def normalize_text(text):
    """Нижний регистр, без знаков ударения, ё -> е."""
    # NFD отделяет ударение от буквы; выбрасываем только ударения, остальное собираем обратно (NFC),
    # иначе вместе с ними пропала бы и кратка в «й»
    text = unicodedata.normalize("NFD", text.lower())
    text = "".join(ch for ch in text if ch not in ACCENTS)
    return unicodedata.normalize("NFC", text).replace("ё", "е")


# словоформ в коллекции конечное число, поэтому разбор каждой кэшируем
@lru_cache(maxsize=None)
def lemma(word):
    return morph.parse(word)[0].normal_form


@lru_cache(maxsize=None)
def stem(word):
    return stemmer.stem(word)


def lemmatize_text(text):
    return " ".join(lemma(w) for w in WORD_RE.findall(normalize_text(text)))


def stem_text(text):
    return " ".join(stem(w) for w in WORD_RE.findall(normalize_text(text)))


print("без чистки:", WORD_RE.findall("Камбо́джа".lower()), "  с чисткой:", WORD_RE.findall(normalize_text("Камбо́джа")))
print(f"абзацев со знаком ударения: {sum(chr(0x301) in t for t in texts)}\n")

for word in ("романа", "написал", "государства", "итмо", "снегга"):
    print(f"{word:>12} -> стем {stem(word):<10} лемма {lemma(word)}")

без чистки: ['камбо', 'джа']   с чисткой: ['камбоджа']
абзацев со знаком ударения: 1026

      романа -> стем рома       лемма роман
     написал -> стем написа     лемма написать
 государства -> стем государств лемма государство
        итмо -> стем итм        лемма итмый
      снегга -> стем снегг      лемма снегга


Лемматизатор ошибается на незнакомых словах: имена, аббревиатуры, транслитерация. Это те самые ошибки вида «Киев → Кий» из лекции про индексацию.

Предобработаем коллекцию один раз и сравним варианты.

In [58]:
start = time.time()
lemma_texts = [lemmatize_text(t) for t in texts]
stem_texts = [stem_text(t) for t in texts]
print(f"предобработка: {time.time() - start:.1f} сек")

prep_variants = {
    "слова как есть": (TfidfVectorizer(sublinear_tf=True), texts, lambda q: q),
    "стемминг": (TfidfVectorizer(sublinear_tf=True), stem_texts, stem_text),
    "лемматизация": (TfidfVectorizer(sublinear_tf=True), lemma_texts, lemmatize_text),
    "символьные 4-граммы": (TfidfVectorizer(sublinear_tf=True, analyzer="char_wb", ngram_range=(4, 4),
                                            preprocessor=normalize_text), texts, lambda q: q),
}

rows = {}
for name, (vec, corpus, prep) in prep_variants.items():
    Xv = vec.fit_transform(corpus)
    rows[name] = {**evaluate(make_ranker(vec, Xv, prep), test), "терминов": Xv.shape[1]}

metrics = ["Recall@10", "MRR@10", "P@10"]
results["TF-IDF, sublinear tf"] = {m: rows["слова как есть"][m] for m in metrics}
results["TF-IDF + символьные 4-граммы"] = {m: rows["символьные 4-граммы"][m] for m in metrics}
results["TF-IDF + лемматизация"] = {m: rows["лемматизация"][m] for m in metrics}
pd.DataFrame(rows).T

предобработка: 9.1 сек


,Recall@10,MRR@10,P@10,терминов
слова как есть,0.495,0.379,0.083,139646.0
стемминг,0.710,0.534,0.118,64049.0
лемматизация,0.713,0.542,0.119,67852.0
символьные 4-граммы,0.670,0.500,0.111,147678.0


Нормализация словоформ (вместе с чисткой ударений) даёт +44% к Recall@10: с 0.495 до 0.71. Для английского такого эффекта нет,
а для русского он огромный — это совпадает с классическими замерами на CLEF (Dolamic & Savoy, 2009), которые мы видели на лекции.

Стемминг и лемматизация здесь практически равны, хотя стеммер тоже ошибается («романа» → «рома»). Символьные 4-граммы отстают на 4 пункта, хотя ничего не знают про язык.
Зато они устойчивы к опечаткам и незнакомым словам.

Дальше работаем с лемматизацией.

In [61]:
lemma_vectorizer = TfidfVectorizer(sublinear_tf=True)
X_lemma = lemma_vectorizer.fit_transform(lemma_texts)
lemma_ranker = make_ranker(lemma_vectorizer, X_lemma, lemmatize_text)

In [68]:
X_lemma.shape

(15010, 67852)

### Стоп-слова

Стоп-слова можно не брать из готового списка, а вычислить: это термины с самой большой документной частотой.

In [62]:
vocab = lemma_vectorizer.get_feature_names_out()
df_lemma = np.bincount(X_lemma.indices, minlength=len(vocab))  # df = в скольких строках матрицы есть термин
by_df = vocab[np.argsort(-df_lemma)]
print("самые частые леммы:", list(by_df[:15]))

rows = {}
for n_stop in (0, 50, 300):
    vec = TfidfVectorizer(sublinear_tf=True, stop_words=list(by_df[:n_stop]) or None)
    rows[f"убрать топ-{n_stop}"] = evaluate(make_ranker(vec, vec.fit_transform(lemma_texts), lemmatize_text), test)
pd.DataFrame(rows).T

самые частые леммы: ['год', 'на', 'быть', 'по', 'из', 'который', 'он', 'не', 'что', 'как', 'от', 'это', 'они', 'для', 'свой']


,Recall@10,MRR@10,P@10
убрать топ-0,0.713,0.542,0.119
убрать топ-50,0.717,0.543,0.120
убрать топ-300,0.687,0.502,0.114


Выкидывание стоп-слов почти ничего не меняет, а агрессивное (топ-300) уже вредит: вместе со служебными словами уходят и полезные.
IDF и так даёт частым словам маленький вес. В ранжировании стоп-слова убирают скорее ради скорости (короче постинг-листы), чем ради качества.

## 5. Кворум: мягкое И

Строгое И на вопросах почти ничего не находит, а ИЛИ тянет тысячи кандидатов. Кворум — компромисс:
оставляем документ, если сумма IDF найденных в нём слов запроса не меньше доли `threshold` от суммарного IDF запроса.

Кандидатов, прошедших кворум, ранжируем как обычно.

In [ ]:
X_lemma_csc = X_lemma.tocsc()  # по столбцам: для термина быстро достаём документы -- это и есть постинг-лист
lemma_idf = lemma_vectorizer.idf_
term_id = lemma_vectorizer.vocabulary_


def quorum_candidates(query_lemmas, threshold):
    # Кворум: документ проходит, если в нём есть слова запроса, суммарно набирающие
    # не меньше threshold от общего IDF запроса (редкие слова весят больше частых)
    terms = [term_id[t] for t in sorted(set(query_lemmas.split())) if t in term_id]
    total = sum(lemma_idf[t] for t in terms)  # "полный вес" запроса
    acc = defaultdict(float)  # документ -> сумма IDF найденных слов
    for t in terms:
        # постинг-лист термина t: номера строк (документов) с ненулевым весом в столбце t
        for doc_num in X_lemma_csc.indices[X_lemma_csc.indptr[t]:X_lemma_csc.indptr[t + 1]]:
            acc[doc_num] += lemma_idf[t]
    return [doc_num for doc_num, s in acc.items() if s >= threshold * total]


def quorum_ranker(threshold):
    def ranker(query, k=10):
        q = lemmatize_text(query)
        cands = quorum_candidates(q, threshold)
        if not cands:
            return []
        scores = (X_lemma[cands] @ lemma_vectorizer.transform([q]).T).toarray().ravel()
        return [doc_ids[cands[i]] for i in np.argsort(-scores, kind="stable")[:k]]
    return ranker


rows = {}
for threshold in (0.0, 0.3, 0.5, 0.7, 1.0):
    n_cands = np.mean([len(quorum_candidates(lemmatize_text(queries[qid]), threshold)) for qid in test])
    rows[threshold] = {"кандидатов в среднем": round(n_cands, 1), **evaluate(quorum_ranker(threshold), test)}
pd.DataFrame(rows).T.rename_axis("порог")

,кандидатов в среднем,Recall@10,MRR@10,P@10
порог,,,,
0.0,3626.6,0.713,0.542,0.119
0.3,92.2,0.701,0.558,0.117
0.5,7.9,0.534,0.509,0.087
0.7,0.9,0.196,0.240,0.032
1.0,0.1,0.004,0.007,0.001


In [66]:
X_lemma_csc.toarray().shape

(15010, 67852)

Порог 0 — это чистое ИЛИ: тысячи кандидатов на запрос. При пороге 0.3 кандидатов в 40 раз меньше, Recall@10 почти не падает,
а MRR@10 даже растёт: кворум выкидывает документы, которые совпали с запросом только по частым словам.
Дальше начинаются потери, а порог 1.0 — это строгое И: почти всегда пусто.

Именно так работает черновой этап в больших поисковиках: дешёвый отбор оставляет сотни документов, и дорогие формулы считаются только для них.

## 6. whoosh: TF-IDF против BM25F

Вернёмся к движку из прошлого семинара. Проиндексируем в него уже лемматизированные тексты, чтобы сравнение было честным,
и поищем с разными формулами ранжирования (`weighting`). Запрос разбираем как ИЛИ.

In [69]:
import shutil
import warnings

warnings.filterwarnings("ignore", category=SyntaxWarning)  # whoosh 2.7.4 старый и шумит на свежем Python

import whoosh.index as whoosh_index
from whoosh import qparser, scoring
from whoosh.analysis import LowercaseFilter, RegexTokenizer
from whoosh.fields import ID, TEXT, Schema

# тексты уже лемматизированы, поэтому анализатор только режет по словам и понижает регистр
analyzer = RegexTokenizer(r"\w+") | LowercaseFilter()
schema = Schema(docid=ID(stored=True), body=TEXT(analyzer=analyzer))

INDEX_DIR = "indexdir_ranking"
shutil.rmtree(INDEX_DIR, ignore_errors=True)
os.mkdir(INDEX_DIR)
ix = whoosh_index.create_in(INDEX_DIR, schema)

start = time.time()
writer = ix.writer(limitmb=512)
for docid, text in zip(doc_ids, lemma_texts):
    writer.add_document(docid=docid, body=text)
writer.commit()
print(f"проиндексировано {ix.doc_count()} документов за {time.time() - start:.1f} сек")

проиндексировано 15010 документов за 4.5 сек


In [ ]:
or_parser = qparser.QueryParser("body", schema, group=qparser.OrGroup)

rows = {}
for name, weighting in [("Frequency (только tf)", scoring.Frequency()),
                        ("TF_IDF", scoring.TF_IDF()),
                        ("BM25F (по умолчанию)", scoring.BM25F())]:
    start = time.time()
    with ix.searcher(weighting=weighting) as searcher:
        def whoosh_ranker(query):
            hits = searcher.search(or_parser.parse(lemmatize_text(query)), limit=10)
            return [hit["docid"] for hit in hits]
        rows[name] = {**evaluate(whoosh_ranker, test), "сек": round(time.time() - start, 1)}

results["whoosh BM25F"] = {k: v for k, v in rows["BM25F (по умолчанию)"].items() if k != "сек"}
pd.DataFrame(rows).T

,Recall@10,MRR@10,P@10,сек
Frequency (только tf),0.128,0.071,0.021,30.6
TF_IDF,0.312,0.167,0.051,30.2
BM25F (по умолчанию),0.727,0.606,0.121,14.5


`TF_IDF` в whoosh — это просто `tf · idf` без нормализации по длине. Он проигрывает даже нашему косинусу:
длинные абзацы набирают больше совпадений и лезут в топ.

**BM25F** заметно лучше по MRR@10 — ответ оказывается выше в топе. В нём есть две вещи, которых нет в простом TF-IDF:
насыщение по tf и аккуратная нормализация длины документа. Откуда это берётся — разберём на лекции «Текстовое ранжирование, часть II».
Поле у нас одно, поэтому `BM25F` здесь — просто BM25: буква F означает несколько полей (зон), о них тоже в части II.

## 7. Лексический разрыв

Посмотрим на запросы, где лучший на сейчас вариант (лемматизация) не нашёл релевантный документ даже в топ-100.

In [71]:
misses = [qid for qid in test if not set(lemma_ranker(queries[qid])[:100]) & qrels[qid]]
print(f"релевантного нет в топ-100: {len(misses)} запросов из {len(test)}\n")

for qid in misses[:4]:
    q_lemmas = set(lemmatize_text(queries[qid]).split())
    docid = sorted(qrels[qid])[0]
    common = q_lemmas & set(lemmatize_text(docs[docid]).split())
    print(queries[qid])
    print("   релевантный:", docs[docid][:140], "...")
    print("   общие леммы:", sorted(common), "\n")

релевантного нет в топ-100: 75 запросов из 1292

Как звали отца Северуса Снейпа?
   релевантный: Происхождение Снегга в основном стало известно из его воспоминаний. Родители Снегга — чистокровная волшебница Эйлин Принц и магл Тобиас Снег ...
   общие леммы: ['как', 'отец'] 

В какой стране находится гора Эверест?
   релевантный: Вершина находится в Гималаях в хребте Махалангур-Химал, по которому проходит граница Непала и Тибетского автономного района (Китай). ...
   общие леммы: ['в', 'находиться'] 

Как называются деньги в Гондурасе?
   релевантный: Национальная валюта — гондурасская лемпира. Введена в обращение в 1931 году и названа в честь индейского вождя. ...
   общие леммы: ['в'] 

Какому музею принадлежит "Джоконда" великого Леонардо?
   релевантный: Король Франциск I, как считается, купил картину у наследников Салаи (за 4 000 экю) и хранил в своём замке Фонтенбло, где она оставалась до в ...
   общие леммы: ['музей'] 



Типичные причины: в документе синоним или другое написание («Снейп» / «Снегг»), слово из вопроса вообще не упоминается
(Эверест описан как «вершина в Гималаях»), а общими остаются только слова вроде «как», «в», «находиться».
Это и есть **lexical mismatch**.

Без чистки ударений из раздела 4 в этот список попадала бы и «столица Камбоджи»: в релевантном абзаце написано «Камбо́джа», и токенизатор резал слово пополам.
Прежде чем списывать промах на синонимы, стоит проверить предобработку.

### 7.1 Псевдорелевантная обратная связь (Rocchio)

Идея: верим, что топ выдачи в основном релевантен. Берём `k` лучших документов и сдвигаем вектор запроса к их центру:
`q' = α·q + β·mean(top-k)`. Запрос получает слова, которых в нём не было, но которые есть в хороших документах.

Параметры подбираем на **dev**.

In [72]:
from scipy import sparse
from sklearn.preprocessing import normalize

doc_num_of = {docid: i for i, docid in enumerate(doc_ids)}


def prf_ranker(k_fb=3, beta=1.0, alpha=1.0):
    def ranker(query, k=100):
        q = lemma_vectorizer.transform([lemmatize_text(query)])
        first = top_k((X_lemma @ q.T).toarray().ravel(), k_fb)           # первый проход
        q_new = alpha * q.toarray() + beta * np.asarray(X_lemma[first].mean(axis=0))
        q_new = sparse.csr_matrix(normalize(q_new))
        return [doc_ids[i] for i in top_k((X_lemma @ q_new.T).toarray().ravel(), k)]  # второй проход
    return ranker


rows = {}
for k_fb in (3, 5, 10):
    for beta in (0.3, 1.0):
        rows[(k_fb, beta)] = evaluate(prf_ranker(k_fb, beta), dev)
print("dev, без PRF:", evaluate(lemma_ranker, dev))
pd.DataFrame(rows).T.rename_axis(["k", "beta"])

dev, без PRF: {'Recall@10': 0.734, 'MRR@10': 0.546, 'P@10': 0.12}


Recall@10  MRR@10   P@10
k  beta                          
3  0.3       0.759   0.547  0.125
   1.0       0.771   0.553  0.128
5  0.3       0.753   0.544  0.124
   1.0       0.770   0.552  0.127
10 0.3       0.738   0.544  0.121
   1.0       0.739   0.545  0.121

In [73]:
results["лемматизация + PRF (k=3, β=1)"] = evaluate(prf_ranker(3, 1.0), test)
results["лемматизация + PRF (k=3, β=1)"]

{'Recall@10': 0.748, 'MRR@10': 0.544, 'P@10': 0.126}

PRF поднимает Recall@10 на 3–4 пункта: запрос обогащается словами из найденных документов. MRR@10 при этом не меняется —
PRF скорее вытаскивает новые документы в топ-10, чем поднимает ответ на первое место. За всё это платим вторым проходом по индексу,
и если первый топ был плохим, расширение уводит запрос не туда (**query drift**). Поэтому `k` маленькое.

### 7.2 LSA

Сожмём матрицу TF-IDF через SVD до 256 «тематик» и будем искать по косинусу плотных векторов.

In [ ]:
from sklearn.decomposition import TruncatedSVD

start = time.time()
svd = TruncatedSVD(n_components=256, random_state=0)
E_docs = normalize(svd.fit_transform(X_lemma))  # плотные эмбеддинги документов
print(f"SVD за {time.time() - start:.1f} сек, эмбеддинги {E_docs.shape}")


def lsa_scores(query):
    e = normalize(svd.transform(lemma_vectorizer.transform([lemmatize_text(query)])))
    return E_docs @ e.ravel()


def lsa_ranker(query, k=100):
    return [doc_ids[i] for i in top_k(lsa_scores(query), k)]


def hybrid_ranker(alpha):
    def ranker(query, k=100):
        sparse_scores = (X_lemma @ lemma_vectorizer.transform([lemmatize_text(query)]).T).toarray().ravel()
        return [doc_ids[i] for i in top_k(alpha * sparse_scores + (1 - alpha) * lsa_scores(query), k)]
    return ranker


print("LSA, test:", evaluate(lsa_ranker, test))
print("dev, без LSA:", evaluate(lemma_ranker, dev))
pd.DataFrame({alpha: evaluate(hybrid_ranker(alpha), dev) for alpha in (0.5, 0.7, 0.9, 0.95)}).T.rename_axis("alpha")

SVD за 2.6 сек, эмбеддинги (15010, 256)
LSA, test: {'Recall@10': 0.228, 'MRR@10': 0.128, 'P@10': 0.036}
dev, без LSA: {'Recall@10': 0.734, 'MRR@10': 0.546, 'P@10': 0.12}


,Recall@10,MRR@10,P@10
alpha,,,
0.50,0.455,0.324,0.074
0.70,0.622,0.479,0.102
0.90,0.742,0.543,0.121
0.95,0.743,0.547,0.121


In [75]:
results["LSA (256)"] = evaluate(lsa_ranker, test)
results["гибрид TF-IDF + LSA (α=0.9)"] = evaluate(hybrid_ranker(0.9), test)
results["гибрид TF-IDF + LSA (α=0.9)"]

{'Recall@10': 0.702, 'MRR@10': 0.547, 'P@10': 0.117}

Сама по себе LSA ищет плохо: 256 тематик слишком грубы, чтобы отличить «столицу Камбоджи» от «столицы Лаоса».
Гибрид на dev выглядел чуть лучше чистого TF-IDF, а на test оказался даже хуже (Recall@10 0.702 против 0.713).
Прирост на dev был шумом — ровно поэтому параметры подбирают на одной выборке, а проверяют на другой.

Это честная картинка: плотные векторы, выученные простым разложением матрицы, лексическому поиску не конкуренты.
Чтобы семантический поиск реально обходил TF-IDF, нужны нейросети, обученные именно на задачу поиска. Это будет в блоке про нейропоиск.

## Сводная таблица (test)

In [76]:
pd.DataFrame(results).T

,Recall@10,MRR@10,P@10
случайная выдача,0.002,0.002,0.000
TF-IDF руками,0.487,0.385,0.082
"TF-IDF, sublinear tf",0.495,0.379,0.083
TF-IDF + символьные 4-граммы,0.670,0.500,0.111
TF-IDF + лемматизация,0.713,0.542,0.119
whoosh BM25F,0.727,0.606,0.121
"лемматизация + PRF (k=3, β=1)",0.748,0.544,0.126
LSA (256),0.228,0.128,0.036
гибрид TF-IDF + LSA (α=0.9),0.702,0.547,0.117


Главное за сегодня:
- без метрики и отложенной выборки нельзя сказать, стало ли лучше;
- IDF и нормализация по длине — обязательная часть формулы;
- для русского нормализация словоформ даёт больше, чем любые тонкие настройки весов, а ошибки предобработки (те же ударения) стоят дороже любой формулы;
- кворум резко сокращает число кандидатов почти без потери качества;
- BM25 из коробки лучше нашего TF-IDF, а расширение запроса (PRF) помогает с лексическим разрывом.

## 8. Задания

**Задание 1. Pivoted normalization.**
Косинус делит вектор документа на его норму `|d|` и этим слишком сильно штрафует длинные документы.
Singhal, Buckley и Mitra (1996) предложили делить на `(1 − s) · pivot + s · |d|`, где `pivot` — средняя норма по коллекции.
Реализуйте это поверх `TfidfVectorizer(sublinear_tf=True, norm=None)` на лемматизированных текстах, подберите `s` на dev и посчитайте метрики на test.
Сравните с косинусом и с BM25F из whoosh.

**Задание 2. Reciprocal Rank Fusion.**
Смешайте две выдачи — лемматизированный TF-IDF и символьные 4-граммы — по формуле RRF: `score(d) = Σ 1 / (60 + rank(d))`.
Для смешивания нужны только позиции, скоры не нужны. Помогает ли? На каких запросах 4-граммы находят то, чего не нашла лемматизация?

**Задание 3. Разбор ошибок.**
Возьмите 10 запросов из test, где у лучшего метода MRR@10 = 0. Для каждого определите причину:
синонимы и перефразирование, ошибка лемматизации, ответ размазан по нескольким абзацам, спорная разметка.
Какая причина самая частая и какой из методов лекции мог бы помочь?

In [26]:
# Задание 1: ваш код

In [27]:
# Задание 2: ваш код

In [28]:
# Задание 3: ваш код